In [1]:
!pip -q install --upgrade openai pandas tqdm

from google.colab import drive
drive.mount("/content/drive")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 138.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 9.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
db-dtypes 1.6.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but yo

In [2]:
import os
import getpass

try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = None

if not api_key:
    api_key = getpass.getpass("OPENAI_API_KEY 입력: ")

os.environ["OPENAI_API_KEY"] = api_key
print("OPENAI_API_KEY 설정 완료")

OPENAI_API_KEY 입력: ··········
OPENAI_API_KEY 설정 완료


In [3]:
# ============================================================
# 인사캠 summary 생성용 경로 설정
# 기존 summary 코드의 DATA_DIR / raw_path / status_path 설정 셀만 교체
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import json
import time
import random
from tqdm.auto import tqdm
from openai import OpenAI

BASE_DIR = Path("/content/drive/MyDrive/Crawler")
DATA_DIR = BASE_DIR / "data" / "h"
OUTPUT_DIR = BASE_DIR / "output" / "h"

RAW_CANDIDATES = [
    "h_raw_reviews_manual.csv",
    "h_02_raw_reviews_manual.csv",
    "raw_reviews_manual.csv",
]

STATUS_CANDIDATES = [
    "h_review_collection_status.csv",
    "h_02_review_collection_status.csv",
    "review_collection_status.csv",
]

def first_existing(base_dir: Path, candidates: list[str]) -> Path:
    for name in candidates:
        path = base_dir / name
        if path.exists():
            return path
    raise FileNotFoundError(f"다음 후보 파일을 찾지 못함: {candidates}")

raw_path = first_existing(DATA_DIR, RAW_CANDIDATES)
status_path = first_existing(DATA_DIR, STATUS_CANDIDATES)

print("DATA_DIR:", DATA_DIR)
print("raw reviews:", raw_path)
print("status:", status_path)

raw_df = pd.read_csv(raw_path, encoding="utf-8-sig")
status_df = pd.read_csv(status_path, encoding="utf-8-sig")

required_raw_cols = [
    "restaurant_external_id",
    "restaurant_name",
    "review_text",
]

required_status_cols = [
    "restaurant_external_id",
    "restaurant_name",
    "collected_review_count",
    "status",
]

for col in required_raw_cols:
    assert col in raw_df.columns, f"raw_reviews_manual.csv에 {col} 컬럼이 없음"

for col in required_status_cols:
    assert col in status_df.columns, f"review_collection_status.csv에 {col} 컬럼이 없음"

print("raw_df shape:", raw_df.shape)
print("status_df shape:", status_df.shape)

display(raw_df.head())
display(status_df.head())

DATA_DIR: /content/drive/MyDrive/Crawler/data/h
raw reviews: /content/drive/MyDrive/Crawler/data/h/h_raw_reviews_manual.csv
status: /content/drive/MyDrive/Crawler/data/h/h_review_collection_status.csv
raw_df shape: (2250, 5)
status_df shape: (45, 6)


,review_external_id,restaurant_external_id,restaurant_name,review_index,review_text
0,kakao_168487509_naver_001,kakao_168487509,인생건어물맥주 대학로점,1,가성비도 좋고 음식도 맛있어요 바로 앞이 성균관대라 바깥 풍경 보면서 먹기 좋아요 :)
1,kakao_168487509_naver_002,kakao_168487509,인생건어물맥주 대학로점,2,노가리 파는줄 알고 갔다가 없어서 오뎅탕 시켰는데 진짜 맛있었어요 칼칼하니 너무 맛...
2,kakao_168487509_naver_003,kakao_168487509,인생건어물맥주 대학로점,3,혼술하기 좋습니다. 직원분들 친절하고요. 건어물은 조금더 구워주셨으면 하는 바램이 ...
3,kakao_168487509_naver_004,kakao_168487509,인생건어물맥주 대학로점,4,대학로 친구 만나러 왔다가 안주맛집 발견!! 상호가 건어물로 되어 있어서 별기대없이...
4,kakao_168487509_naver_005,kakao_168487509,인생건어물맥주 대학로점,5,바삭한 먹태를 마요네즈 소스에 찍어먹으니 정말 맛있어요~


,restaurant_external_id,restaurant_name,kakao_place_id,kakao_place_url,collected_review_count,status
0,kakao_168487509,인생건어물맥주 대학로점,168487509,http://place.map.kakao.com/168487509,13,done
1,kakao_11634881,한솥도시락 성균관대앞점,11634881,http://place.map.kakao.com/11634881,50,done
2,kakao_22016023,맥덕스,22016023,http://place.map.kakao.com/22016023,50,done
3,kakao_338935651,명륜진사갈비 서울성균관대점,338935651,http://place.map.kakao.com/338935651,50,done
4,kakao_122821905,물결식당,122821905,http://place.map.kakao.com/122821905,15,done


In [4]:
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\u00a0", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

raw_df["review_text_clean"] = raw_df["review_text"].apply(clean_text)

raw_nonempty = raw_df[raw_df["review_text_clean"] != ""].copy()

reviews_map = (
    raw_nonempty
    .sort_values(["restaurant_external_id", "review_index"])
    .groupby("restaurant_external_id")["review_text_clean"]
    .apply(list)
    .to_dict()
)

# 현재 두 CSV에는 리뷰별 신뢰도 점수 컬럼이 없을 가능성이 높음.
# 향후 review_trust_score / trust_score / credibility_score 등이 추가되면 자동 평균 계산.
TRUST_SCORE_CANDIDATES = [
    "average_trust_score",
    "review_trust_score",
    "trust_score",
    "credibility_score",
    "final_score",
    "score",
    "non_sponsored_probability",
]

trust_col = None
for c in TRUST_SCORE_CANDIDATES:
    if c in raw_df.columns:
        trust_col = c
        break

avg_trust_map = {}

if trust_col:
    raw_df[trust_col] = pd.to_numeric(raw_df[trust_col], errors="coerce")

    # 0~1 확률값이면 0~100 점수로 변환
    max_val = raw_df[trust_col].max(skipna=True)
    if pd.notna(max_val) and max_val <= 1.0:
        raw_df[trust_col] = raw_df[trust_col] * 100.0

    avg_trust_map = raw_df.groupby("restaurant_external_id")[trust_col].mean().to_dict()
    print("신뢰도 점수 컬럼 사용:", trust_col)
else:
    print("신뢰도 점수 컬럼 없음: average_trust_score는 NAN으로 출력함")

print("리뷰가 있는 식당 수:", len(reviews_map))

신뢰도 점수 컬럼 없음: average_trust_score는 NAN으로 출력함
리뷰가 있는 식당 수: 37


In [5]:
client = OpenAI()


MODEL = "gpt-5.5"

OUTPUT_COLUMNS = [
    "restaurant_external_id",
    "summary",
    "representative_menu",
    "mood_summary",
    "parking_summary",
    "waiting_summary",
    "average_trust_score",
    "has_analysis",
]

SUMMARY_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "summary": {
            "type": "string",
            "description": "리뷰 기반 식당 전체 요약. 반드시 한국어로 작성한다."
        },
        "representative_menu": {
            "anyOf": [{"type": "string"}, {"type": "null"}],
            "description": "리뷰에서 확인되는 대표 메뉴. 여러 개면 | 로 연결. 없으면 null."
        },
        "mood_summary": {
            "anyOf": [{"type": "string"}, {"type": "null"}],
            "description": "리뷰에서 확인되는 분위기 요약. 없으면 null."
        },
        "parking_summary": {
            "anyOf": [{"type": "string"}, {"type": "null"}],
            "description": "리뷰에서 확인되는 주차 정보. 없으면 null."
        },
        "waiting_summary": {
            "anyOf": [{"type": "string"}, {"type": "null"}],
            "description": "리뷰에서 확인되는 웨이팅 정보. 없으면 null."
        },
    },
    "required": [
        "summary",
        "representative_menu",
        "mood_summary",
        "parking_summary",
        "waiting_summary",
    ],
}

SYSTEM_PROMPT = """
너는 성균관대학교 주변 맛집 추천 서비스의 리뷰 요약 모듈이다.
입력으로 한 식당의 실제 방문자 리뷰 목록을 받는다.

반드시 지킬 규칙:
1. 리뷰에 근거한 정보만 추출한다.
2. summary는 반드시 작성한다.
3. representative_menu, mood_summary, parking_summary, waiting_summary는 리뷰에 명시적 근거가 없으면 null로 반환한다.
4. '정보 없음', '언급 없음', '알 수 없음' 같은 문장을 쓰지 말고 null을 사용한다.
5. 주차와 웨이팅은 특히 추측하지 않는다.
6. 대표 메뉴는 메뉴명만 간결하게 쓴다. 여러 개면 | 로 연결한다.
7. 모든 출력은 JSON Schema에 맞춘다.
"""

def normalize_optional_value(x):
    if x is None:
        return np.nan

    x = str(x).strip()

    invalid_values = {
        "",
        "nan",
        "NaN",
        "NAN",
        "null",
        "None",
        "없음",
        "정보 없음",
        "언급 없음",
        "해당 없음",
        "알 수 없음",
    }

    if x in invalid_values:
        return np.nan

    return x

def build_user_prompt(restaurant_name, reviews):
    # 리뷰가 너무 긴 경우 비용과 토큰 초과 방지를 위해 리뷰별 500자까지만 사용
    clipped_reviews = []
    for i, review in enumerate(reviews, start=1):
        review = clean_text(review)
        if len(review) > 500:
            review = review[:500] + "..."
        clipped_reviews.append(f"{i}. {review}")

    reviews_text = "\n".join(clipped_reviews)

    return f"""
식당명: {restaurant_name}

리뷰 목록:
{reviews_text}

작업:
아래 5개 필드를 추출하라.

1. summary
- 필수
- 리뷰 전체를 바탕으로 식당의 특징을 1~2문장으로 요약
- 맛, 서비스, 가격, 재방문 의사, 이용 목적 등을 종합
- 과장하지 말고 리뷰에 나온 내용만 반영

2. representative_menu
- 리뷰에서 실제로 언급된 메뉴 또는 추천 메뉴
- 여러 개면 메뉴명만 | 로 연결
- 메뉴 정보가 없으면 null

3. mood_summary
- 분위기, 인테리어, 소음, 데이트, 혼밥, 모임, 술자리 등 리뷰에 언급된 분위기 정보
- 관련 정보가 없으면 null

4. parking_summary
- 주차 가능 여부, 주차 어려움, 근처 주차 등 리뷰에 언급된 주차 정보
- 관련 정보가 없으면 null

5. waiting_summary
- 웨이팅 있음, 자리 기다림, 회전율, 대기시간 등 리뷰에 언급된 웨이팅 정보
- 관련 정보가 없으면 null
"""

def extract_with_gpt(restaurant_name, reviews, max_retries=5):
    user_prompt = build_user_prompt(restaurant_name, reviews)

    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model=MODEL,
                reasoning={
                    "effort": "low"
                },
                max_output_tokens=1200,
                input=[
                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT,
                    },
                    {
                        "role": "user",
                        "content": user_prompt,
                    },
                ],
                text={
                    "format": {
                        "type": "json_schema",
                        "name": "restaurant_review_summary",
                        "strict": True,
                        "schema": SUMMARY_SCHEMA,
                    }
                },
            )

            data = json.loads(response.output_text)

            summary = str(data.get("summary", "")).strip()
            if not summary:
                summary = f"{restaurant_name}은 리뷰 기반으로 음식과 방문 경험을 요약할 수 있는 식당입니다."

            return {
                "summary": summary,
                "representative_menu": normalize_optional_value(data.get("representative_menu")),
                "mood_summary": normalize_optional_value(data.get("mood_summary")),
                "parking_summary": normalize_optional_value(data.get("parking_summary")),
                "waiting_summary": normalize_optional_value(data.get("waiting_summary")),
            }

        except Exception as e:
            err_text = str(e)

            if "billing_not_active" in err_text:
                raise RuntimeError(
                    "OpenAI API billing이 아직 활성화되지 않았습니다. "
                    "결제수단/크레딧 활성화 후 다시 실행하세요."
                )

            wait = min(60, (2 ** attempt) + random.random())
            print(f"[재시도 {attempt + 1}/{max_retries}] {restaurant_name}: {e}")
            time.sleep(wait)

    return {
        "summary": f"{restaurant_name}은 리뷰 요약 API 호출 실패로 인해 자동 요약을 완료하지 못했습니다.",
        "representative_menu": np.nan,
        "mood_summary": np.nan,
        "parking_summary": np.nan,
        "waiting_summary": np.nan,
    }

In [6]:
# ==========================================
# Cell 6. restaurant_summaries.csv 처음부터 생성 + 한 줄씩 즉시 저장
# ==========================================

import csv
import math
from tqdm.auto import tqdm

# 테스트할 때는 1 또는 3
# 전체 실행할 때는 None
LIMIT = None

output_path = DATA_DIR / "restaurant_summaries.csv"

target_status_df = status_df.copy()

if LIMIT is not None:
    target_status_df = target_status_df.head(LIMIT)

def to_csv_value(x):
    """
    CSV에 쓸 값 정리.
    NaN/None은 요구사항에 맞게 NAN 문자열로 저장.
    """
    if x is None:
        return "NAN"

    try:
        if pd.isna(x):
            return "NAN"
    except Exception:
        pass

    if isinstance(x, float) and math.isnan(x):
        return "NAN"

    return x

# 기존 파일 삭제/초기화 후 헤더부터 새로 작성
with open(output_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=OUTPUT_COLUMNS)
    writer.writeheader()

print("새 restaurant_summaries.csv 생성 시작:", output_path)
print("대상 식당 수:", len(target_status_df))

for _, row in tqdm(target_status_df.iterrows(), total=len(target_status_df)):
    restaurant_external_id = str(row["restaurant_external_id"])
    restaurant_name = str(row["restaurant_name"])

    reviews = reviews_map.get(restaurant_external_id, [])
    has_reviews = len(reviews) > 0

    avg_trust_score = avg_trust_map.get(restaurant_external_id, np.nan)

    if has_reviews:
        extracted = extract_with_gpt(restaurant_name, reviews)

        # GPT 호출 실패 fallback 문구면 분석 실패로 처리
        if "API 호출 실패" in str(extracted.get("summary", "")):
            has_analysis = "false"
        else:
            has_analysis = "true"

    else:
        extracted = {
            "summary": f"{restaurant_name}은 수집된 리뷰가 없어 리뷰 기반 상세 요약을 생성할 수 없습니다.",
            "representative_menu": np.nan,
            "mood_summary": np.nan,
            "parking_summary": np.nan,
            "waiting_summary": np.nan,
        }
        has_analysis = "false"

    out_row = {
        "restaurant_external_id": restaurant_external_id,
        "summary": extracted["summary"],
        "representative_menu": extracted["representative_menu"],
        "mood_summary": extracted["mood_summary"],
        "parking_summary": extracted["parking_summary"],
        "waiting_summary": extracted["waiting_summary"],
        "average_trust_score": avg_trust_score,
        "has_analysis": has_analysis,
    }

    out_row = {
        col: to_csv_value(out_row.get(col))
        for col in OUTPUT_COLUMNS
    }

    # 한 식당 끝날 때마다 바로 파일에 append
    with open(output_path, "a", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_COLUMNS)
        writer.writerow(out_row)

print("restaurant_summaries.csv 생성 완료:", output_path)

summaries_df = pd.read_csv(output_path, encoding="utf-8-sig")

print("shape:", summaries_df.shape)
display(summaries_df.head(10))

새 restaurant_summaries.csv 생성 시작: /content/drive/MyDrive/Crawler/data/h/restaurant_summaries.csv
대상 식당 수: 45


  0%|          | 0/45 [00:00<?, ?it/s]

restaurant_summaries.csv 생성 완료: /content/drive/MyDrive/Crawler/data/h/restaurant_summaries.csv
shape: (45, 8)


,restaurant_external_id,summary,representative_menu,mood_summary,parking_summary,waiting_summary,average_trust_score,has_analysis
0,kakao_168487509,"성균관대 바로 앞에 있어 위치가 좋고, 안주와 맥주가 맛있으며 가격도 저렴하다는 평...",오뎅탕 | 마라바지락볶음면 | 감바스 | 먹태 | 살얼음생맥주,"성균관대 앞 풍경을 보며 먹기 좋고, 혼술이나 친구와 대화하며 술 마시기 좋은 분위...",NAN,NAN,NAN,True
1,kakao_11634881,"저렴한 가격과 가성비 좋은 요일·행사 메뉴, 맛있는 도시락으로 점심이나 간단한 식사...",불닭치킨마요 | 치즈닭갈비덮밥 | 스팸마요 | 치킨마요 | 순살 후라이드,혼밥하기 좋고 편하게 간단한 식사를 하기 좋은 매장입니다.,NAN,"도시락이나 포장이 빨리 나온다는 리뷰가 있지만, 주문 후 음식이 나오기까지 조금 시...",NAN,True
2,kakao_22016023,"성균관대 앞에서 피자와 맥주 조합을 즐기기 좋은 피맥집으로, 피자 도우와 다양한 수...",페퍼로니 피자 | 비비큐 포크 피자 | 더블 치즈 피자 | 맥앤치즈 | 치즈 오븐 ...,"돌담길 옆 야외 테라스와 야장 감성이 좋고, 아늑하고 조용한 분위기라 데이트나 피맥...",NAN,NAN,NAN,True
3,kakao_338935651,"고기 질과 양념갈비 맛, 다양한 사이드 메뉴와 가성비에 대한 만족도가 높고 직원·사...",돼지갈비 | 양념갈비 | 돼지껍데기 | 계란찜 | 된장찌개 | 햄버거 | 감자튀김 ...,"매장이 넓고 깔끔하며 단체석이 많아 가족모임, 회식, 친구 모임, 데이트에 이용하기...",주차가 편하고 주차 공간이 넓거나 넉넉하다는 리뷰가 여러 번 언급되었습니다.,"예약 손님이 많아 자리를 못 잡을 뻔했다는 언급과, 주말에는 인원이 있으면 예약이 ...",NAN,True
4,kakao_122821905,"물결식당은 고등어덮밥, 카레, 닭고기 덮밥 등 덮밥류와 파스타 메뉴가 맛있다는 평가...",고등어덮밥 | 카레 | 쟌슨빌 크림 라이스 | 닭고기 덮밥 | 잔슨빌들깨파스타 | ...,"아담하고 아기자기한 인테리어에 조용하고 따뜻한 분위기이며, 일본 감성이나 연남동 골...",NAN,NAN,NAN,True
5,kakao_25770908,"피자가 맛있고 양이 많다는 평가가 많으며, 가성비가 좋고 자주 배달시켜 먹는다는 리...",해븐피자 | 사대천왕 피자 | 고구마스위트 | 눈꽃치즈스테이크 | 더블치즈베이컨 |...,생일 파티에 주문하기 좋았다는 언급이 있습니다.,NAN,NAN,NAN,True
6,kakao_1869234515,"피자와 치킨, 떡강정 등을 맛있게 먹었다는 리뷰가 많고 도우가 얇고 쫄깃하다는 언급...",케이준 떡강정 | 케이준 양념감자 | 핫치즈빅싸이순살 | 빅싸이순살 치킨 | 싸이피...,"아이들과 간단히 먹기 좋고, 혼술하기 좋을 것 같다는 언급이 있습니다.",NAN,NAN,NAN,True
7,kakao_483030266,"성균관대 근처의 빵집으로 소금빵이 특히 많이 언급되며, 버터향과 짭짤함이 좋고 가격...",소금빵 | 브라우니 | 휘낭시에 | 치아바타 앙버터 | 에그타르트 | 밀크프레즐 |...,매장이 작고 협소하며 테이크아웃 위주로 이용하기 좋아 보인다는 언급이 있습니다. 빵...,NAN,NAN,NAN,True
8,kakao_633176459,"안주와 음식이 전반적으로 맛있고 종류가 다양하다는 평가가 많으며, 사장님과 직원들이...",짬뽕탕 | 먹태 | 치킨 | 곱창전골 | 명란 크림 우동 | 술찜 | 돈가쓰 | 크...,"대화하기 좋고 분위기가 좋다는 평가가 있으며, 인테리어가 예쁘고 좌석이 편하다는 언...",NAN,NAN,NAN,True
9,kakao_11520636,"성균관대 앞에서 오래 사랑받는 이란·인도식 커리 맛집으로, 통닭카레와 케밥, 난, ...",통닭정식 | 통닭카레 | 치킨커리 | 양고기케밥 | 양고기카레 | 버터난 | 치즈난...,"중동·이란·인도 현지 느낌의 이국적인 인테리어와 독특한 분위기가 강하게 언급되며, ...",NAN,"대기가 있는 편이라는 리뷰가 많으며, 브레이크타임 직후나 오픈런에는 바로 앉거나 비...",NAN,True


In [7]:
# ==========================================
# Cell 7. restaurant_summaries.csv 검증
# ==========================================

from pathlib import Path
import pandas as pd
import numpy as np

output_path = DATA_DIR / "restaurant_summaries.csv"
error_path = DATA_DIR / "restaurant_summaries_validation_errors.csv"

assert output_path.exists(), f"파일이 존재하지 않음: {output_path}"

# NAN 문자열을 그대로 보기 위해 keep_default_na=False 사용
summaries_df = pd.read_csv(
    output_path,
    encoding="utf-8-sig",
    keep_default_na=False,
)

print("파일 경로:", output_path)
print("shape:", summaries_df.shape)
print("columns:", summaries_df.columns.tolist())

errors = []

# ------------------------------------------
# 1. 컬럼 검증
# ------------------------------------------

if list(summaries_df.columns) != OUTPUT_COLUMNS:
    errors.append({
        "type": "column_mismatch",
        "restaurant_external_id": "ALL",
        "message": f"컬럼 불일치. 현재={summaries_df.columns.tolist()}, 필요={OUTPUT_COLUMNS}",
    })

# ------------------------------------------
# 2. 행 개수 검증
# ------------------------------------------

expected_total = len(status_df)
actual_total = len(summaries_df)

if actual_total != expected_total:
    errors.append({
        "type": "row_count_mismatch",
        "restaurant_external_id": "ALL",
        "message": f"행 개수 불일치. 현재={actual_total}, 기대={expected_total}. LIMIT 테스트 중이면 정상일 수 있음.",
    })

# ------------------------------------------
# 3. restaurant_external_id 검증
# ------------------------------------------

if "restaurant_external_id" in summaries_df.columns:
    summaries_df["restaurant_external_id"] = summaries_df["restaurant_external_id"].astype(str)
    status_ids = set(status_df["restaurant_external_id"].astype(str))
    summary_ids = set(summaries_df["restaurant_external_id"].astype(str))

    duplicated_ids = summaries_df[
        summaries_df["restaurant_external_id"].duplicated(keep=False)
    ]["restaurant_external_id"].unique().tolist()

    for rid in duplicated_ids:
        errors.append({
            "type": "duplicate_restaurant_external_id",
            "restaurant_external_id": rid,
            "message": "restaurant_external_id 중복",
        })

    missing_ids = sorted(list(status_ids - summary_ids))
    extra_ids = sorted(list(summary_ids - status_ids))

    for rid in missing_ids:
        errors.append({
            "type": "missing_restaurant_external_id",
            "restaurant_external_id": rid,
            "message": "status_df에는 있으나 restaurant_summaries.csv에는 없음",
        })

    for rid in extra_ids:
        errors.append({
            "type": "extra_restaurant_external_id",
            "restaurant_external_id": rid,
            "message": "restaurant_summaries.csv에는 있으나 status_df에는 없음",
        })

# ------------------------------------------
# 4. 필수값 검증
# ------------------------------------------

required_nonempty_cols = [
    "restaurant_external_id",
    "summary",
    "has_analysis",
]

for col in required_nonempty_cols:
    if col in summaries_df.columns:
        bad = summaries_df[
            summaries_df[col].astype(str).str.strip().isin(["", "NAN", "nan", "NaN", "None", "null"])
        ]

        for _, r in bad.iterrows():
            errors.append({
                "type": f"empty_required_{col}",
                "restaurant_external_id": r.get("restaurant_external_id", "UNKNOWN"),
                "message": f"{col} 필수값이 비어 있음",
            })

# ------------------------------------------
# 5. has_analysis 값 검증
# ------------------------------------------

if "has_analysis" in summaries_df.columns:
    valid_bool_strings = {"true", "false", "True", "False", True, False}

    bad = summaries_df[
        ~summaries_df["has_analysis"].isin(valid_bool_strings)
    ]

    for _, r in bad.iterrows():
        errors.append({
            "type": "invalid_has_analysis",
            "restaurant_external_id": r.get("restaurant_external_id", "UNKNOWN"),
            "message": f"has_analysis 값이 true/false가 아님: {r.get('has_analysis')}",
        })

# ------------------------------------------
# 6. 선택 필드 검증
# optional 필드는 NAN 허용
# ------------------------------------------

optional_cols = [
    "representative_menu",
    "mood_summary",
    "parking_summary",
    "waiting_summary",
]

for col in optional_cols:
    if col in summaries_df.columns:
        # 완전 빈 문자열은 DB import 전에 NAN으로 맞추는 것이 안전함
        bad = summaries_df[
            summaries_df[col].astype(str).str.strip() == ""
        ]

        for _, r in bad.iterrows():
            errors.append({
                "type": f"empty_optional_should_be_NAN_{col}",
                "restaurant_external_id": r.get("restaurant_external_id", "UNKNOWN"),
                "message": f"{col}이 빈 문자열임. NAN으로 저장하는 것이 안전함.",
            })

# ------------------------------------------
# 7. average_trust_score 검증
# NAN 또는 0~100 숫자 허용
# ------------------------------------------

if "average_trust_score" in summaries_df.columns:
    for _, r in summaries_df.iterrows():
        rid = r.get("restaurant_external_id", "UNKNOWN")
        value = str(r.get("average_trust_score", "")).strip()

        if value in ["", "NAN", "nan", "NaN", "None", "null"]:
            continue

        try:
            score = float(value)
            if score < 0 or score > 100:
                errors.append({
                    "type": "average_trust_score_out_of_range",
                    "restaurant_external_id": rid,
                    "message": f"average_trust_score가 0~100 범위를 벗어남: {value}",
                })
        except Exception:
            errors.append({
                "type": "invalid_average_trust_score",
                "restaurant_external_id": rid,
                "message": f"average_trust_score가 숫자 또는 NAN이 아님: {value}",
            })

# ------------------------------------------
# 8. summary 품질 간단 검증
# ------------------------------------------

if "summary" in summaries_df.columns:
    for _, r in summaries_df.iterrows():
        rid = r.get("restaurant_external_id", "UNKNOWN")
        summary = str(r.get("summary", "")).strip()

        if len(summary) < 10:
            errors.append({
                "type": "summary_too_short",
                "restaurant_external_id": rid,
                "message": f"summary가 너무 짧음: {summary}",
            })

        if "API 호출 실패" in summary:
            errors.append({
                "type": "api_fallback_summary",
                "restaurant_external_id": rid,
                "message": "GPT API 실패 fallback summary가 포함됨",
            })

# ------------------------------------------
# 9. 결과 출력
# ------------------------------------------

errors_df = pd.DataFrame(errors)

if len(errors_df) == 0:
    print("검증 통과: restaurant_summaries.csv가 기본 스키마 조건을 만족함")
else:
    print("검증 실패 또는 경고:", len(errors_df), "건")
    errors_df.to_csv(error_path, index=False, encoding="utf-8-sig")
    print("검증 오류 저장:", error_path)
    display(errors_df)

print("\n미리보기")
display(summaries_df.head(10))

print("\nNAN 개수 확인")
nan_report = pd.DataFrame({
    "column": summaries_df.columns,
    "NAN_count": [
        summaries_df[col].astype(str).str.strip().isin(["NAN", "nan", "NaN", "", "None", "null"]).sum()
        for col in summaries_df.columns
    ],
    "non_NAN_count": [
        len(summaries_df) - summaries_df[col].astype(str).str.strip().isin(["NAN", "nan", "NaN", "", "None", "null"]).sum()
        for col in summaries_df.columns
    ],
})
display(nan_report)

파일 경로: /content/drive/MyDrive/Crawler/data/h/restaurant_summaries.csv
shape: (45, 8)
columns: ['restaurant_external_id', 'summary', 'representative_menu', 'mood_summary', 'parking_summary', 'waiting_summary', 'average_trust_score', 'has_analysis']
검증 통과: restaurant_summaries.csv가 기본 스키마 조건을 만족함

미리보기


,restaurant_external_id,summary,representative_menu,mood_summary,parking_summary,waiting_summary,average_trust_score,has_analysis
0,kakao_168487509,"성균관대 바로 앞에 있어 위치가 좋고, 안주와 맥주가 맛있으며 가격도 저렴하다는 평...",오뎅탕 | 마라바지락볶음면 | 감바스 | 먹태 | 살얼음생맥주,"성균관대 앞 풍경을 보며 먹기 좋고, 혼술이나 친구와 대화하며 술 마시기 좋은 분위...",NAN,NAN,NAN,True
1,kakao_11634881,"저렴한 가격과 가성비 좋은 요일·행사 메뉴, 맛있는 도시락으로 점심이나 간단한 식사...",불닭치킨마요 | 치즈닭갈비덮밥 | 스팸마요 | 치킨마요 | 순살 후라이드,혼밥하기 좋고 편하게 간단한 식사를 하기 좋은 매장입니다.,NAN,"도시락이나 포장이 빨리 나온다는 리뷰가 있지만, 주문 후 음식이 나오기까지 조금 시...",NAN,True
2,kakao_22016023,"성균관대 앞에서 피자와 맥주 조합을 즐기기 좋은 피맥집으로, 피자 도우와 다양한 수...",페퍼로니 피자 | 비비큐 포크 피자 | 더블 치즈 피자 | 맥앤치즈 | 치즈 오븐 ...,"돌담길 옆 야외 테라스와 야장 감성이 좋고, 아늑하고 조용한 분위기라 데이트나 피맥...",NAN,NAN,NAN,True
3,kakao_338935651,"고기 질과 양념갈비 맛, 다양한 사이드 메뉴와 가성비에 대한 만족도가 높고 직원·사...",돼지갈비 | 양념갈비 | 돼지껍데기 | 계란찜 | 된장찌개 | 햄버거 | 감자튀김 ...,"매장이 넓고 깔끔하며 단체석이 많아 가족모임, 회식, 친구 모임, 데이트에 이용하기...",주차가 편하고 주차 공간이 넓거나 넉넉하다는 리뷰가 여러 번 언급되었습니다.,"예약 손님이 많아 자리를 못 잡을 뻔했다는 언급과, 주말에는 인원이 있으면 예약이 ...",NAN,True
4,kakao_122821905,"물결식당은 고등어덮밥, 카레, 닭고기 덮밥 등 덮밥류와 파스타 메뉴가 맛있다는 평가...",고등어덮밥 | 카레 | 쟌슨빌 크림 라이스 | 닭고기 덮밥 | 잔슨빌들깨파스타 | ...,"아담하고 아기자기한 인테리어에 조용하고 따뜻한 분위기이며, 일본 감성이나 연남동 골...",NAN,NAN,NAN,True
5,kakao_25770908,"피자가 맛있고 양이 많다는 평가가 많으며, 가성비가 좋고 자주 배달시켜 먹는다는 리...",해븐피자 | 사대천왕 피자 | 고구마스위트 | 눈꽃치즈스테이크 | 더블치즈베이컨 |...,생일 파티에 주문하기 좋았다는 언급이 있습니다.,NAN,NAN,NAN,True
6,kakao_1869234515,"피자와 치킨, 떡강정 등을 맛있게 먹었다는 리뷰가 많고 도우가 얇고 쫄깃하다는 언급...",케이준 떡강정 | 케이준 양념감자 | 핫치즈빅싸이순살 | 빅싸이순살 치킨 | 싸이피...,"아이들과 간단히 먹기 좋고, 혼술하기 좋을 것 같다는 언급이 있습니다.",NAN,NAN,NAN,True
7,kakao_483030266,"성균관대 근처의 빵집으로 소금빵이 특히 많이 언급되며, 버터향과 짭짤함이 좋고 가격...",소금빵 | 브라우니 | 휘낭시에 | 치아바타 앙버터 | 에그타르트 | 밀크프레즐 |...,매장이 작고 협소하며 테이크아웃 위주로 이용하기 좋아 보인다는 언급이 있습니다. 빵...,NAN,NAN,NAN,True
8,kakao_633176459,"안주와 음식이 전반적으로 맛있고 종류가 다양하다는 평가가 많으며, 사장님과 직원들이...",짬뽕탕 | 먹태 | 치킨 | 곱창전골 | 명란 크림 우동 | 술찜 | 돈가쓰 | 크...,"대화하기 좋고 분위기가 좋다는 평가가 있으며, 인테리어가 예쁘고 좌석이 편하다는 언...",NAN,NAN,NAN,True
9,kakao_11520636,"성균관대 앞에서 오래 사랑받는 이란·인도식 커리 맛집으로, 통닭카레와 케밥, 난, ...",통닭정식 | 통닭카레 | 치킨커리 | 양고기케밥 | 양고기카레 | 버터난 | 치즈난...,"중동·이란·인도 현지 느낌의 이국적인 인테리어와 독특한 분위기가 강하게 언급되며, ...",NAN,"대기가 있는 편이라는 리뷰가 많으며, 브레이크타임 직후나 오픈런에는 바로 앉거나 비...",NAN,True



NAN 개수 확인


,column,NAN_count,non_NAN_count
0,restaurant_external_id,0,45
1,summary,0,45
2,representative_menu,8,37
3,mood_summary,8,37
4,parking_summary,44,1
5,waiting_summary,31,14
6,average_trust_score,45,0
7,has_analysis,0,45


In [8]:
# ==========================================
# 인사캠 average_trust_score 주입
# Crawler/output/h/h_review_credibility_restaurant_scores.csv
# -> Crawler/data/h/restaurant_summaries.csv
# ==========================================

from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path("/content/drive/MyDrive/Crawler")
DATA_DIR = BASE_DIR / "data" / "h"
OUTPUT_DIR = BASE_DIR / "output" / "h"

summary_path = DATA_DIR / "restaurant_summaries.csv"

score_candidates = [
    OUTPUT_DIR / "h_review_credibility_restaurant_scores.csv",
    OUTPUT_DIR / "review_credibility_restaurant_scores.csv",
]

score_path = None
for p in score_candidates:
    if p.exists():
        score_path = p
        break

if score_path is None:
    raise FileNotFoundError(
        "인사캠 review_credibility_restaurant_scores.csv 파일을 찾지 못했습니다."
    )

print("summary_path:", summary_path)
print("score_path:", score_path)

summaries_df = pd.read_csv(summary_path, encoding="utf-8-sig", keep_default_na=False)
scores_df = pd.read_csv(score_path, encoding="utf-8-sig")

required_score_cols = [
    "restaurant_external_id",
    "average_trust_score",
]

for col in required_score_cols:
    assert col in scores_df.columns, f"score 파일에 {col} 컬럼이 없음"

assert "restaurant_external_id" in summaries_df.columns
assert "average_trust_score" in summaries_df.columns

summaries_df["restaurant_external_id"] = summaries_df["restaurant_external_id"].astype(str)
scores_df["restaurant_external_id"] = scores_df["restaurant_external_id"].astype(str)

scores_df["average_trust_score"] = pd.to_numeric(
    scores_df["average_trust_score"],
    errors="coerce"
)

score_map = (
    scores_df
    .drop_duplicates(subset=["restaurant_external_id"], keep="first")
    .set_index("restaurant_external_id")["average_trust_score"]
    .to_dict()
)

summaries_df["average_trust_score"] = summaries_df["restaurant_external_id"].map(score_map)
summaries_df["average_trust_score"] = summaries_df["average_trust_score"].fillna(0.0)
summaries_df["average_trust_score"] = summaries_df["average_trust_score"].round(2)

summaries_df = summaries_df[OUTPUT_COLUMNS]

summaries_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
    na_rep="NAN",
)

print("average_trust_score 주입 완료")
print("저장 완료:", summary_path)

display(summaries_df.head(10))

summary_path: /content/drive/MyDrive/Crawler/data/h/restaurant_summaries.csv
score_path: /content/drive/MyDrive/Crawler/output/h/h_review_credibility_restaurant_scores.csv
average_trust_score 주입 완료
저장 완료: /content/drive/MyDrive/Crawler/data/h/restaurant_summaries.csv


,restaurant_external_id,summary,representative_menu,mood_summary,parking_summary,waiting_summary,average_trust_score,has_analysis
0,kakao_168487509,"성균관대 바로 앞에 있어 위치가 좋고, 안주와 맥주가 맛있으며 가격도 저렴하다는 평...",오뎅탕 | 마라바지락볶음면 | 감바스 | 먹태 | 살얼음생맥주,"성균관대 앞 풍경을 보며 먹기 좋고, 혼술이나 친구와 대화하며 술 마시기 좋은 분위...",NAN,NAN,32.03,True
1,kakao_11634881,"저렴한 가격과 가성비 좋은 요일·행사 메뉴, 맛있는 도시락으로 점심이나 간단한 식사...",불닭치킨마요 | 치즈닭갈비덮밥 | 스팸마요 | 치킨마요 | 순살 후라이드,혼밥하기 좋고 편하게 간단한 식사를 하기 좋은 매장입니다.,NAN,"도시락이나 포장이 빨리 나온다는 리뷰가 있지만, 주문 후 음식이 나오기까지 조금 시...",44.68,True
2,kakao_22016023,"성균관대 앞에서 피자와 맥주 조합을 즐기기 좋은 피맥집으로, 피자 도우와 다양한 수...",페퍼로니 피자 | 비비큐 포크 피자 | 더블 치즈 피자 | 맥앤치즈 | 치즈 오븐 ...,"돌담길 옆 야외 테라스와 야장 감성이 좋고, 아늑하고 조용한 분위기라 데이트나 피맥...",NAN,NAN,30.48,True
3,kakao_338935651,"고기 질과 양념갈비 맛, 다양한 사이드 메뉴와 가성비에 대한 만족도가 높고 직원·사...",돼지갈비 | 양념갈비 | 돼지껍데기 | 계란찜 | 된장찌개 | 햄버거 | 감자튀김 ...,"매장이 넓고 깔끔하며 단체석이 많아 가족모임, 회식, 친구 모임, 데이트에 이용하기...",주차가 편하고 주차 공간이 넓거나 넉넉하다는 리뷰가 여러 번 언급되었습니다.,"예약 손님이 많아 자리를 못 잡을 뻔했다는 언급과, 주말에는 인원이 있으면 예약이 ...",32.76,True
4,kakao_122821905,"물결식당은 고등어덮밥, 카레, 닭고기 덮밥 등 덮밥류와 파스타 메뉴가 맛있다는 평가...",고등어덮밥 | 카레 | 쟌슨빌 크림 라이스 | 닭고기 덮밥 | 잔슨빌들깨파스타 | ...,"아담하고 아기자기한 인테리어에 조용하고 따뜻한 분위기이며, 일본 감성이나 연남동 골...",NAN,NAN,21.17,True
5,kakao_25770908,"피자가 맛있고 양이 많다는 평가가 많으며, 가성비가 좋고 자주 배달시켜 먹는다는 리...",해븐피자 | 사대천왕 피자 | 고구마스위트 | 눈꽃치즈스테이크 | 더블치즈베이컨 |...,생일 파티에 주문하기 좋았다는 언급이 있습니다.,NAN,NAN,44.31,True
6,kakao_1869234515,"피자와 치킨, 떡강정 등을 맛있게 먹었다는 리뷰가 많고 도우가 얇고 쫄깃하다는 언급...",케이준 떡강정 | 케이준 양념감자 | 핫치즈빅싸이순살 | 빅싸이순살 치킨 | 싸이피...,"아이들과 간단히 먹기 좋고, 혼술하기 좋을 것 같다는 언급이 있습니다.",NAN,NAN,49.57,True
7,kakao_483030266,"성균관대 근처의 빵집으로 소금빵이 특히 많이 언급되며, 버터향과 짭짤함이 좋고 가격...",소금빵 | 브라우니 | 휘낭시에 | 치아바타 앙버터 | 에그타르트 | 밀크프레즐 |...,매장이 작고 협소하며 테이크아웃 위주로 이용하기 좋아 보인다는 언급이 있습니다. 빵...,NAN,NAN,32.27,True
8,kakao_633176459,"안주와 음식이 전반적으로 맛있고 종류가 다양하다는 평가가 많으며, 사장님과 직원들이...",짬뽕탕 | 먹태 | 치킨 | 곱창전골 | 명란 크림 우동 | 술찜 | 돈가쓰 | 크...,"대화하기 좋고 분위기가 좋다는 평가가 있으며, 인테리어가 예쁘고 좌석이 편하다는 언...",NAN,NAN,37.32,True
9,kakao_11520636,"성균관대 앞에서 오래 사랑받는 이란·인도식 커리 맛집으로, 통닭카레와 케밥, 난, ...",통닭정식 | 통닭카레 | 치킨커리 | 양고기케밥 | 양고기카레 | 버터난 | 치즈난...,"중동·이란·인도 현지 느낌의 이국적인 인테리어와 독특한 분위기가 강하게 언급되며, ...",NAN,"대기가 있는 편이라는 리뷰가 많으며, 브레이크타임 직후나 오픈런에는 바로 앉거나 비...",31.80,True


In [9]:
# ==========================================
# Cell 9. average_trust_score 주입 확인
# ==========================================

check_df = pd.read_csv(
    DATA_DIR / "restaurant_summaries.csv",
    encoding="utf-8-sig",
    keep_default_na=False,
)

print(check_df.shape)
display(check_df[[
    "restaurant_external_id",
    "summary",
    "average_trust_score",
    "has_analysis",
]].head(10))

print("\naverage_trust_score 통계")
display(check_df["average_trust_score"].astype(float).describe())

print("\nNAN 개수 확인")
nan_report = pd.DataFrame({
    "column": check_df.columns,
    "NAN_count": [
        check_df[col].astype(str).str.strip().isin(["NAN", "nan", "NaN", "", "None", "null"]).sum()
        for col in check_df.columns
    ],
    "non_NAN_count": [
        len(check_df) - check_df[col].astype(str).str.strip().isin(["NAN", "nan", "NaN", "", "None", "null"]).sum()
        for col in check_df.columns
    ],
})
display(nan_report)

(45, 8)


,restaurant_external_id,summary,average_trust_score,has_analysis
0,kakao_168487509,"성균관대 바로 앞에 있어 위치가 좋고, 안주와 맥주가 맛있으며 가격도 저렴하다는 평...",32.03,True
1,kakao_11634881,"저렴한 가격과 가성비 좋은 요일·행사 메뉴, 맛있는 도시락으로 점심이나 간단한 식사...",44.68,True
2,kakao_22016023,"성균관대 앞에서 피자와 맥주 조합을 즐기기 좋은 피맥집으로, 피자 도우와 다양한 수...",30.48,True
3,kakao_338935651,"고기 질과 양념갈비 맛, 다양한 사이드 메뉴와 가성비에 대한 만족도가 높고 직원·사...",32.76,True
4,kakao_122821905,"물결식당은 고등어덮밥, 카레, 닭고기 덮밥 등 덮밥류와 파스타 메뉴가 맛있다는 평가...",21.17,True
5,kakao_25770908,"피자가 맛있고 양이 많다는 평가가 많으며, 가성비가 좋고 자주 배달시켜 먹는다는 리...",44.31,True
6,kakao_1869234515,"피자와 치킨, 떡강정 등을 맛있게 먹었다는 리뷰가 많고 도우가 얇고 쫄깃하다는 언급...",49.57,True
7,kakao_483030266,"성균관대 근처의 빵집으로 소금빵이 특히 많이 언급되며, 버터향과 짭짤함이 좋고 가격...",32.27,True
8,kakao_633176459,"안주와 음식이 전반적으로 맛있고 종류가 다양하다는 평가가 많으며, 사장님과 직원들이...",37.32,True
9,kakao_11520636,"성균관대 앞에서 오래 사랑받는 이란·인도식 커리 맛집으로, 통닭카레와 케밥, 난, ...",31.80,True



average_trust_score 통계


count    45.000000
mean     31.539111
std      16.366179
min       0.000000
25%      28.910000
50%      35.020000
75%      43.030000
max      54.900000
Name: average_trust_score, dtype: float64


NAN 개수 확인


,column,NAN_count,non_NAN_count
0,restaurant_external_id,0,45
1,summary,0,45
2,representative_menu,8,37
3,mood_summary,8,37
4,parking_summary,44,1
5,waiting_summary,31,14
6,average_trust_score,0,45
7,has_analysis,0,45


In [10]:
# ==========================================
# 인사캠 restaurant_summaries.csv -> restaurant_tags.csv 생성
# GPT-5.5 사용
# 기존 tag 생성 코드에서 경로 설정부만 이걸로 교체
# ==========================================

import csv
import json
import time
import random
import math
import re
from pathlib import Path

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-5.5"

BASE_DIR = Path("/content/drive/MyDrive/Crawler")
DATA_DIR = BASE_DIR / "data" / "h"

summary_path = DATA_DIR / "restaurant_summaries.csv"
tag_output_path = DATA_DIR / "restaurant_tags.csv"

print("summary_path:", summary_path)
print("tag_output_path:", tag_output_path)

summary_path: /content/drive/MyDrive/Crawler/data/h/restaurant_summaries.csv
tag_output_path: /content/drive/MyDrive/Crawler/data/h/restaurant_tags.csv


In [11]:
from pathlib import Path

H_DATA_DIR = Path("/content/drive/MyDrive/Crawler/data/h")

print("H_DATA_DIR:", H_DATA_DIR)
print("exists:", H_DATA_DIR.exists())

for p in sorted(H_DATA_DIR.glob("*.csv")):
    print(p.name)

H_DATA_DIR: /content/drive/MyDrive/Crawler/data/h
exists: True
h_kakao_restaurant_candidates.csv
h_raw_reviews_manual.csv
h_review_collection_status.csv
restaurant_price_category_enrichment.csv
restaurant_summaries.csv


In [12]:
# ==========================================
# Tag extraction schema / prompt / helper
# ==========================================

TAGS_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "tags": {
            "type": "array",
            "description": "식당 태그 목록. 태그가 없으면 빈 배열.",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "tag_type": {
                        "type": "string",
                        "enum": [
                            "food",
                            "mood",
                            "price",
                            "purpose",
                            "parking",
                            "waiting",
                        ],
                    },
                    "tag_value": {
                        "type": "string",
                        "description": "한국어 태그 값. 문장이 아니라 짧은 명사구.",
                    },
                    "confidence_score": {
                        "type": "number",
                        "minimum": 0,
                        "maximum": 1,
                    },
                },
                "required": [
                    "tag_type",
                    "tag_value",
                    "confidence_score",
                ],
            },
        }
    },
    "required": [
        "tags",
    ],
}

TAG_SYSTEM_PROMPT = """
너는 성균관대학교 주변 맛집 추천 서비스의 태그 생성 모듈이다.
입력으로 restaurant_summaries.csv의 식당별 요약 정보를 받는다.

목표:
restaurant_tags.csv에 들어갈 태그 목록을 생성한다.

반드시 지킬 규칙:
1. tag_type은 food, mood, price, purpose, parking, waiting 중 하나만 사용한다.
2. tag_value는 한국어 짧은 명사구로 작성한다.
3. tag_value에 긴 문장, 조사 많은 설명문, 마침표를 넣지 않는다.
4. 입력에 근거가 있는 태그만 생성한다.
5. NAN, 없음, 정보 없음, 언급 없음 같은 값을 태그로 만들지 않는다.
6. 주차와 웨이팅은 parking_summary, waiting_summary에 명확한 정보가 있을 때만 만든다.
7. 대표 메뉴는 representative_menu에 있는 메뉴를 food 태그로 만든다.
8. 여러 메뉴가 | 로 연결되어 있으면 각각 별도 food 태그로 만든다.
9. price 태그는 가격, 가성비, 저렴함, 비쌈 같은 근거가 summary에 있을 때만 만든다.
10. purpose 태그는 혼밥, 데이트, 모임, 회식, 술자리, 포장, 가족식사처럼 방문 목적이 드러날 때만 만든다.
11. 한 식당당 태그는 최대 10개까지만 만든다.
12. 같은 tag_type과 tag_value 조합을 중복 생성하지 않는다.

confidence_score 기준:
- representative_menu에서 직접 나온 food 태그: 0.95
- mood_summary에서 명확한 분위기 태그: 0.90
- summary/mood_summary에서 명확한 purpose 태그: 0.90
- parking_summary에서 명확한 parking 태그: 0.85
- waiting_summary에서 명확한 waiting 태그: 0.85
- price 관련 태그: 0.80 ~ 0.90
"""

def is_nan_like(x):
    if x is None:
        return True

    x = str(x).strip()

    return x in {
        "",
        "NAN",
        "nan",
        "NaN",
        "None",
        "null",
        "NULL",
        "없음",
        "정보 없음",
        "언급 없음",
        "해당 없음",
        "알 수 없음",
    }

def clean_tag_value(x):
    x = str(x).strip()
    x = re.sub(r"\s+", "", x)
    x = x.replace(",", "")
    x = x.replace(".", "")
    x = x.replace(";", "")
    x = x.replace(":", "")
    x = x.replace("|", "")
    return x.strip()

def build_tag_user_prompt(row):
    def v(col):
        value = row.get(col, "NAN")
        if is_nan_like(value):
            return "NAN"
        return str(value).strip()

    return f"""
아래 식당 요약 정보를 바탕으로 restaurant_tags.csv에 들어갈 태그를 생성하라.

restaurant_external_id:
{v("restaurant_external_id")}

summary:
{v("summary")}

representative_menu:
{v("representative_menu")}

mood_summary:
{v("mood_summary")}

parking_summary:
{v("parking_summary")}

waiting_summary:
{v("waiting_summary")}

has_analysis:
{v("has_analysis")}

태그 유형별 의미:
food = 대표 메뉴 또는 음식 종류
mood = 분위기
price = 가격대 또는 가성비
purpose = 방문 목적
parking = 주차 정보
waiting = 웨이팅 정보

출력:
JSON Schema에 맞춰 tags 배열만 반환하라.
"""

def normalize_tags(tags):
    normalized = []
    seen = set()

    valid_types = {
        "food",
        "mood",
        "price",
        "purpose",
        "parking",
        "waiting",
    }

    for tag in tags:
        tag_type = str(tag.get("tag_type", "")).strip()
        tag_value = clean_tag_value(tag.get("tag_value", ""))
        confidence_score = tag.get("confidence_score", 0.8)

        if tag_type not in valid_types:
            continue

        if is_nan_like(tag_value):
            continue

        if len(tag_value) < 2:
            continue

        try:
            confidence_score = float(confidence_score)
        except Exception:
            confidence_score = 0.8

        confidence_score = max(0.0, min(1.0, confidence_score))
        confidence_score = round(confidence_score, 2)

        key = (tag_type, tag_value)
        if key in seen:
            continue

        seen.add(key)

        normalized.append({
            "tag_type": tag_type,
            "tag_value": tag_value,
            "confidence_score": confidence_score,
        })

    return normalized[:10]

def extract_tags_with_gpt(row, max_retries=5):
    user_prompt = build_tag_user_prompt(row)

    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model=MODEL,
                reasoning={
                    "effort": "low"
                },
                max_output_tokens=1200,
                input=[
                    {
                        "role": "system",
                        "content": TAG_SYSTEM_PROMPT,
                    },
                    {
                        "role": "user",
                        "content": user_prompt,
                    },
                ],
                text={
                    "format": {
                        "type": "json_schema",
                        "name": "restaurant_tag_extraction",
                        "strict": True,
                        "schema": TAGS_SCHEMA,
                    }
                },
            )

            data = json.loads(response.output_text)
            tags = data.get("tags", [])
            return normalize_tags(tags)

        except Exception as e:
            err_text = str(e)

            if "billing_not_active" in err_text:
                raise RuntimeError(
                    "OpenAI API billing이 활성화되지 않았습니다. "
                    "Billing에서 결제수단 또는 credit을 확인하세요."
                )

            wait = min(60, (2 ** attempt) + random.random())
            rid = row.get("restaurant_external_id", "UNKNOWN")
            print(f"[재시도 {attempt + 1}/{max_retries}] {rid}: {e}")
            time.sleep(wait)

    return []

In [14]:
# ==========================================
# 누락 변수 복구 셀
# ==========================================

import csv
import json
import time
import random
import re
from pathlib import Path

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from openai import OpenAI

try:
    client
except NameError:
    client = OpenAI()

MODEL = "gpt-5.5"

BASE_DIR = Path("/content/drive/MyDrive/Crawler")
DATA_DIR = BASE_DIR / "data" / "h"

summary_path = DATA_DIR / "restaurant_summaries.csv"
tag_output_path = DATA_DIR / "restaurant_tags.csv"

TAG_OUTPUT_COLUMNS = [
    "restaurant_external_id",
    "tag_type",
    "tag_value",
    "confidence_score",
]

assert summary_path.exists(), f"restaurant_summaries.csv 없음: {summary_path}"

summaries_df = pd.read_csv(
    summary_path,
    encoding="utf-8-sig",
    keep_default_na=False,
)

print("summary_path:", summary_path)
print("tag_output_path:", tag_output_path)
print("summaries_df shape:", summaries_df.shape)
print("TAG_OUTPUT_COLUMNS:", TAG_OUTPUT_COLUMNS)

display(summaries_df.head())

summary_path: /content/drive/MyDrive/Crawler/data/h/restaurant_summaries.csv
tag_output_path: /content/drive/MyDrive/Crawler/data/h/restaurant_tags.csv
summaries_df shape: (45, 8)
TAG_OUTPUT_COLUMNS: ['restaurant_external_id', 'tag_type', 'tag_value', 'confidence_score']


,restaurant_external_id,summary,representative_menu,mood_summary,parking_summary,waiting_summary,average_trust_score,has_analysis
0,kakao_168487509,"성균관대 바로 앞에 있어 위치가 좋고, 안주와 맥주가 맛있으며 가격도 저렴하다는 평...",오뎅탕 | 마라바지락볶음면 | 감바스 | 먹태 | 살얼음생맥주,"성균관대 앞 풍경을 보며 먹기 좋고, 혼술이나 친구와 대화하며 술 마시기 좋은 분위...",NAN,NAN,32.03,True
1,kakao_11634881,"저렴한 가격과 가성비 좋은 요일·행사 메뉴, 맛있는 도시락으로 점심이나 간단한 식사...",불닭치킨마요 | 치즈닭갈비덮밥 | 스팸마요 | 치킨마요 | 순살 후라이드,혼밥하기 좋고 편하게 간단한 식사를 하기 좋은 매장입니다.,NAN,"도시락이나 포장이 빨리 나온다는 리뷰가 있지만, 주문 후 음식이 나오기까지 조금 시...",44.68,True
2,kakao_22016023,"성균관대 앞에서 피자와 맥주 조합을 즐기기 좋은 피맥집으로, 피자 도우와 다양한 수...",페퍼로니 피자 | 비비큐 포크 피자 | 더블 치즈 피자 | 맥앤치즈 | 치즈 오븐 ...,"돌담길 옆 야외 테라스와 야장 감성이 좋고, 아늑하고 조용한 분위기라 데이트나 피맥...",NAN,NAN,30.48,True
3,kakao_338935651,"고기 질과 양념갈비 맛, 다양한 사이드 메뉴와 가성비에 대한 만족도가 높고 직원·사...",돼지갈비 | 양념갈비 | 돼지껍데기 | 계란찜 | 된장찌개 | 햄버거 | 감자튀김 ...,"매장이 넓고 깔끔하며 단체석이 많아 가족모임, 회식, 친구 모임, 데이트에 이용하기...",주차가 편하고 주차 공간이 넓거나 넉넉하다는 리뷰가 여러 번 언급되었습니다.,"예약 손님이 많아 자리를 못 잡을 뻔했다는 언급과, 주말에는 인원이 있으면 예약이 ...",32.76,True
4,kakao_122821905,"물결식당은 고등어덮밥, 카레, 닭고기 덮밥 등 덮밥류와 파스타 메뉴가 맛있다는 평가...",고등어덮밥 | 카레 | 쟌슨빌 크림 라이스 | 닭고기 덮밥 | 잔슨빌들깨파스타 | ...,"아담하고 아기자기한 인테리어에 조용하고 따뜻한 분위기이며, 일본 감성이나 연남동 골...",NAN,NAN,21.17,True


In [15]:
# ==========================================
# 인사캠 restaurant_tags.csv 생성 실행
# ==========================================

# 테스트할 때는 1 또는 3
# 전체 실행할 때는 None
LIMIT = None

target_df = summaries_df.copy()

if LIMIT is not None:
    target_df = target_df.head(LIMIT)

# 기존 파일 삭제 후 새로 생성
with open(tag_output_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=TAG_OUTPUT_COLUMNS)
    writer.writeheader()

print("restaurant_tags.csv 생성 시작:", tag_output_path)
print("대상 식당 수:", len(target_df))

total_tag_count = 0

for _, row in tqdm(target_df.iterrows(), total=len(target_df)):
    restaurant_external_id = str(row["restaurant_external_id"]).strip()

    tags = extract_tags_with_gpt(row)

    tag_rows = []
    for tag in tags:
        tag_rows.append({
            "restaurant_external_id": restaurant_external_id,
            "tag_type": tag["tag_type"],
            "tag_value": tag["tag_value"],
            "confidence_score": tag["confidence_score"],
        })

    if len(tag_rows) > 0:
        with open(tag_output_path, "a", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=TAG_OUTPUT_COLUMNS)
            writer.writerows(tag_rows)

    total_tag_count += len(tag_rows)

print("restaurant_tags.csv 생성 완료:", tag_output_path)
print("총 태그 수:", total_tag_count)

tags_df = pd.read_csv(tag_output_path, encoding="utf-8-sig")
print("shape:", tags_df.shape)
display(tags_df.head(30))

restaurant_tags.csv 생성 시작: /content/drive/MyDrive/Crawler/data/h/restaurant_tags.csv
대상 식당 수: 45


  0%|          | 0/45 [00:00<?, ?it/s]

restaurant_tags.csv 생성 완료: /content/drive/MyDrive/Crawler/data/h/restaurant_tags.csv
총 태그 수: 357
shape: (357, 4)


,restaurant_external_id,tag_type,tag_value,confidence_score
0,kakao_168487509,food,오뎅탕,0.95
1,kakao_168487509,food,마라바지락볶음면,0.95
2,kakao_168487509,food,감바스,0.95
3,kakao_168487509,food,먹태,0.95
4,kakao_168487509,food,살얼음생맥주,0.95
5,kakao_168487509,mood,예쁜인테리어,0.90
6,kakao_168487509,mood,깔끔한분위기,0.90
7,kakao_168487509,price,저렴한가격,0.85
8,kakao_168487509,purpose,혼술,0.90
9,kakao_168487509,purpose,친구술자리,0.90
